# Module 2 – Data Cleaning & Transformation

## Project
MedTrack_DV (Hospital Operations & Patient Analytics Dashboard)

### Intern
Sanika Patil

### Objective
Clean and preprocess the raw hospital dataset to create a Tableau-ready dataset for KPI engineering and dashboard development.

Import Libraries

In [90]:
import pandas as pd
import numpy as np

In [91]:
df = pd.read_csv("Hospital_RawDataset.csv")

df["Admission Date"] = pd.to_datetime(
    df["Admission Date"],
    format="mixed"
)

df["Discharge Date"] = pd.to_datetime(
    df["Discharge Date"],
    format="mixed"
)

print(df["Admission Date"].isna().sum())
print(df["Discharge Date"].isna().sum())

0
0


Dataset Overview

In [92]:
df.head()
df.shape
df.info()
df.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 49 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Hospital ID                     10000 non-null  str           
 1   Hospital Name                   10000 non-null  str           
 2   Hospital Type                   10000 non-null  str           
 3   City                            10000 non-null  str           
 4   State                           10000 non-null  str           
 5   Department                      10000 non-null  str           
 6   Department ID                   10000 non-null  str           
 7   Doctor                          10000 non-null  str           
 8   Nurses                          10000 non-null  int64         
 9   Staff Count                     10000 non-null  int64         
 10  Patient ID                      10000 non-null  str           
 11  Patient Name  

,Hospital ID,Hospital Name,Hospital Type,City,State,Department,Department ID,Doctor,Nurses,Staff Count,...,Transferred,Transfer_From_Department,Transfer_To_Department,Transfer_Date,Number_of_Transfers,Dept_ICU_Bed_Capacity_Derived,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%
count,10000,10000,10000,10000,10000,10000,10000,10000,10000.00000,10000.000000,...,10000,10000,1760,1760,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
unique,20,20,2,6,6,10,10,4,NaN,NaN,...,2,10,10,523,NaN,NaN,NaN,NaN,NaN,NaN
top,HOSP00012,Medindia Hospital,Private,Chennai,Tamil Nadu,Neurology,DEPT002,Dr. Rahul Verma,NaN,NaN,...,No,Neurology,Psychiatry,2026-03-02 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
freq,532,532,8496,1706,1706,1033,1033,2556,NaN,NaN,...,8240,1033,190,13,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.53610,341.421000,...,NaN,NaN,NaN,NaN,0.245800,98.701000,589.466100,0.110160,57.924860,0.36757
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.00000,80.000000,...,NaN,NaN,NaN,NaN,0.000000,86.000000,554.000000,0.000000,13.300000,0.20000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.00000,214.750000,...,NaN,NaN,NaN,NaN,0.000000,98.000000,584.000000,0.000000,36.375000,0.20000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.00000,343.000000,...,NaN,NaN,NaN,NaN,0.000000,99.000000,593.000000,0.200000,58.200000,0.40000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.00000,470.000000,...,NaN,NaN,NaN,NaN,0.000000,100.000000,598.000000,0.200000,79.600000,0.40000
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.00000,600.000000,...,NaN,NaN,NaN,NaN,3.000000,100.000000,600.000000,0.600000,100.000000,1.40000


Checking Missing Values

The columns with missing values were inspected individually to determine whether the missing values represented genuine missing information or valid business cases (e.g., patients not transferred).

In [93]:
missing = df.isnull().sum()

missing[missing > 0]

Insurance Provider        2463
Transfer_To_Department    8240
Transfer_Date             8240
dtype: int64

In [94]:
df["Transfer_To_Department"].value_counts(dropna=False)

Transfer_To_Department
NaN                 8240
Psychiatry           190
Oncology             188
Orthopedics          187
ICU                  185
Nephrology           181
Neurology            176
General Surgery      174
Cardiology           168
Pulmonology          159
General Medicine     152
Name: count, dtype: int64

In [95]:
df["Transfer_Date"].value_counts(dropna=False)

Transfer_Date
NaN                    8240
2026-03-02 00:00:00      13
5/24/2026                 9
6/27/2025                 9
1/27/2026                 9
                       ... 
2/15/2026                 1
2025-05-01 00:00:00       1
2025-06-11 00:00:00       1
2025-11-12 00:00:00       1
2026-05-05 00:00:00       1
Name: count, Length: 524, dtype: int64

In [96]:
df["Insurance Provider"].value_counts(dropna=False)

Insurance Provider
Niva Bupa        2536
Star Health      2509
ICICI Lombard    2492
NaN              2463
Name: count, dtype: int64

In [97]:
df["Insurance Provider"] = df["Insurance Provider"].fillna("Self-Pay")

df["Transfer_To_Department"] = df["Transfer_To_Department"].fillna("No Transfer")

In [98]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Transfer_Date    8240
dtype: int64


Removing Duplicate Records

In [99]:
duplicates = df.duplicated().sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


No duplicate rows were found

Standardizing Text Columns

In [100]:
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].str.strip()

C:\Users\admin\AppData\Local\Temp\ipykernel_18656\3659800821.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


In [101]:
df.head()

,Hospital ID,Hospital Name,Hospital Type,City,State,Department,Department ID,Doctor,Nurses,Staff Count,...,Transferred,Transfer_From_Department,Transfer_To_Department,Transfer_Date,Number_of_Transfers,Dept_ICU_Bed_Capacity_Derived,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%
0,HOSP00016,C K Hospital,Private,Mumbai,Maharashtra,General Medicine,DEPT004,Dr. Arjun Sharma,12,257,...,No,General Medicine,No Transfer,NaN,0,98,598,0.0,43.0,0.2
1,HOSP00019,Cure Well Hospital,Private,Bengaluru,Karnataka,General Surgery,DEPT010,Dr. Sneha Rao,15,288,...,No,General Surgery,No Transfer,NaN,0,100,583,0.2,49.4,0.2
2,HOSP00004,Yashoda Hospital,Private,Hyderabad,Telangana,Pulmonology,DEPT005,Dr. Priya Reddy,20,181,...,No,Pulmonology,No Transfer,NaN,0,97,590,0.0,30.7,0.2
3,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Pulmonology,DEPT005,Dr. Rahul Verma,23,170,...,Yes,Pulmonology,Cardiology,9/17/2025,1,99,597,0.2,28.5,0.4
4,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,ICU,DEPT009,Dr. Priya Reddy,34,116,...,No,ICU,No Transfer,NaN,0,100,576,0.0,20.1,0.6


Verifying datatypes

In [102]:
df["Admission Date"] = pd.to_datetime(df["Admission Date"], format="mixed")
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"], format="mixed")
df["Transfer_Date"] = pd.to_datetime(df["Transfer_Date"], format="mixed")

df.dtypes

Hospital ID                                  str
Hospital Name                                str
Hospital Type                                str
City                                         str
State                                        str
Department                                   str
Department ID                                str
Doctor                                       str
Nurses                                     int64
Staff Count                                int64
Patient ID                                   str
Patient Name                                 str
Gender                                       str
Age                                        int64
Admission Date                    datetime64[us]
Discharge Date                    datetime64[us]
Diagnosis                                    str
Treatment                                    str
Medication                                   str
Admission Type                               str
Test Result         

Outlier detection

In [103]:
df.describe()

,Nurses,Staff Count,Age,Admission Date,Discharge Date,Beds Available,ICU Beds,Bed Number,Length of Stay,Room No,...,Beds_Occupied_Count,Equipment_Total_Inventory,Equipment_Usage_Duration_Hours,Transfer_Date,Number_of_Transfers,Dept_ICU_Bed_Capacity_Derived,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%
count,10000.00000,10000.000000,10000.000000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,...,10000.000000,10000.000000,10000.000000,1760,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,27.53610,341.421000,45.303900,2025-10-08 00:55:17.760000,2025-10-13 07:26:49.920000,258.893300,52.675900,251.115300,8.068400,550.325600,...,1.835500,13.262000,35.859760,2025-10-12 15:08:10.909091,0.245800,98.701000,589.466100,0.110160,57.924860,0.36757
min,5.00000,80.000000,1.000000,2025-01-01 00:00:00,2025-01-02 00:00:00,20.000000,5.000000,1.000000,1.000000,100.000000,...,1.000000,2.000000,0.500000,2025-01-02 00:00:00,0.000000,86.000000,554.000000,0.000000,13.300000,0.20000
25%,16.00000,214.750000,23.000000,2025-05-17 00:00:00,2025-05-21 00:00:00,137.000000,29.000000,128.000000,4.000000,326.000000,...,1.000000,8.000000,18.100000,2025-05-19 00:00:00,0.000000,98.000000,584.000000,0.000000,36.375000,0.20000
50%,28.00000,343.000000,45.000000,2025-09-27 00:00:00,2025-10-03 00:00:00,258.000000,53.000000,250.000000,8.000000,550.000000,...,2.000000,13.000000,35.700000,2025-10-03 12:00:00,0.000000,99.000000,593.000000,0.200000,58.200000,0.40000
75%,39.00000,470.000000,68.000000,2026-02-19 00:00:00,2026-02-26 00:00:00,381.000000,76.000000,377.000000,12.000000,776.250000,...,2.000000,19.000000,53.500000,2026-02-24 00:00:00,0.000000,100.000000,598.000000,0.200000,79.600000,0.40000
max,50.00000,600.000000,90.000000,2026-12-06 00:00:00,2026-12-06 00:00:00,500.000000,100.000000,500.000000,15.000000,999.000000,...,7.000000,24.000000,72.000000,2026-12-06 00:00:00,3.000000,100.000000,600.000000,0.600000,100.000000,1.40000
std,13.24947,149.201603,25.952659,NaN,NaN,139.770105,27.745581,143.325642,4.309708,258.519784,...,0.917019,6.436404,20.567494,NaN,0.595834,1.743704,10.151047,0.109845,25.296697,0.18417


In [104]:
df.to_csv("Hospital_CleanedDataset.csv", index=False)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


# Conclusion

The hospital dataset was successfully cleaned and prepared for analysis.

The following preprocessing tasks were completed:

- Missing values handled appropriately
- Duplicate records checked
- Text values standardized
- Date columns verified
- Data types validated
- Clean dataset exported as Hospital_CleanedDataset.csv

The cleaned dataset is now ready for KPI calculation and Tableau dashboard development.

In [2]:
import pandas as pd

df = pd.read_csv("Hospital_CleanedDataset.csv")

In [3]:
df[df["Transfer_Date"].isna()][["Transfer_To_Department", "Transfer_Date"]].head(20)

,Transfer_To_Department,Transfer_Date
0,No Transfer,NaN
1,No Transfer,NaN
2,No Transfer,NaN
4,No Transfer,NaN
5,No Transfer,NaN
6,No Transfer,NaN
7,No Transfer,NaN
8,No Transfer,NaN
10,No Transfer,NaN
11,No Transfer,NaN


In [4]:
df["Transfer_Date"] = df["Transfer_Date"].astype("object")
df["Transfer_Date"] = df["Transfer_Date"].fillna("No Transfer Happened")

In [5]:
print(df["Transfer_Date"].isnull().sum())

0


In [6]:
df["Transfer_Date"].head(20)

0     No Transfer Happened
1     No Transfer Happened
2     No Transfer Happened
3               2025-09-17
4     No Transfer Happened
5     No Transfer Happened
6     No Transfer Happened
7     No Transfer Happened
8     No Transfer Happened
9               2025-04-27
10    No Transfer Happened
11    No Transfer Happened
12    No Transfer Happened
13    No Transfer Happened
14    No Transfer Happened
15    No Transfer Happened
16              2026-01-06
17    No Transfer Happened
18    No Transfer Happened
19    No Transfer Happened
Name: Transfer_Date, dtype: object

In [7]:
df.to_csv("Hospital_CleanedDataset.csv", index=False)

## Mentor Feedback Update

As per mentor feedback, missing values in the `Transfer_Date` column were replaced with `"No Transfer Happened"` for patients who were not transferred.